In [1]:
from paddleocr import PaddleOCR
from pathlib import Path
import pandas as pd
import json
from tqdm import tqdm


def extract_texts_from_paddle_result(result):
    texts = []
    confidences = []

    if result is None:
        return texts, confidences

    # 兼容 PaddleOCR 新旧不同输出格式
    try:
        # 新版可能是 dict
        if isinstance(result, dict):
            if "rec_texts" in result:
                for text, score in zip(result.get("rec_texts", []), result.get("rec_scores", [])):
                    if text:
                        texts.append(text)
                        confidences.append(float(score))
            return texts, confidences

        # 旧版通常是 list
        for page in result:
            if page is None:
                continue

            if isinstance(page, dict):
                if "rec_texts" in page:
                    for text, score in zip(page.get("rec_texts", []), page.get("rec_scores", [])):
                        if text:
                            texts.append(text)
                            confidences.append(float(score))
                continue

            for line in page:
                if line is None:
                    continue

                # 旧格式: [box, (text, confidence)]
                if isinstance(line, (list, tuple)) and len(line) >= 2:
                    info = line[1]

                    if isinstance(info, (list, tuple)) and len(info) >= 2:
                        text = info[0]
                        score = info[1]

                        if text:
                            texts.append(str(text))
                            confidences.append(float(score))

    except Exception as e:
        print("解析 OCR 结果失败:", e)

    return texts, confidences


def generate_pseudo_labels(
    folder_path,
    output_csv="pseudo_labels.csv",
    output_json="pseudo_labels.json",
    exts=(".png", ".jpg", ".jpeg", ".bmp", ".webp"),
    min_confidence=0.3
):
    folder_path = Path(folder_path)

    image_paths = [
        p for p in folder_path.iterdir()
        if p.suffix.lower() in exts
    ]

    if len(image_paths) == 0:
        raise ValueError(f"文件夹中没有找到图片: {folder_path}")

    ocr = PaddleOCR(lang="ch")

    records = []
    image_label_dict = {}

    for img_path in tqdm(image_paths, desc="Generating pseudo labels"):
        try:
            result = ocr.ocr(str(img_path))

            detected_texts, confidences = extract_texts_from_paddle_result(result)

            valid = [
                (text, conf)
                for text, conf in zip(detected_texts, confidences)
                if conf >= min_confidence and len(text.strip()) > 0
            ]

            if len(valid) > 0:
                raw_text = "".join([v[0] for v in valid])
                label = raw_text[0] if len(raw_text) > 0 else "UNKNOWN"
                confidence = max([v[1] for v in valid])
            else:
                label = "UNKNOWN"
                raw_text = ""
                confidence = 0.0

            image_label_dict[img_path.name] = label

            records.append({
                "filename": img_path.name,
                "path": str(img_path),
                "pseudo_label": label,
                "raw_ocr_text": raw_text,
                "confidence": confidence,
                "error": ""
            })

        except Exception as e:
            records.append({
                "filename": img_path.name,
                "path": str(img_path),
                "pseudo_label": "ERROR",
                "raw_ocr_text": "",
                "confidence": 0.0,
                "error": str(e)
            })

    df = pd.DataFrame(records)

    df.to_csv(output_csv, index=False, encoding="utf-8-sig")

    with open(output_json, "w", encoding="utf-8") as f:
        json.dump(image_label_dict, f, ensure_ascii=False, indent=2)

    print("=" * 60)
    print("Pseudo-label generation finished")
    print(f"Total images: {len(df)}")
    print(f"Saved CSV: {output_csv}")
    print(f"Saved JSON: {output_json}")
    print("=" * 60)

    return df, image_label_dict

C:\Anaconda\lib\site-packages\scipy\__init__.py:138: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.24.4)
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion} is required for this version of "
C:\Anaconda\lib\site-packages\pydantic\_internal\_fields.py:161: UserWarning: Field "model_kwargs" has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(


In [ ]:
# choose the target data in the folder and name the output files accordingly
folder_path = "./calli-kaggle/data/data/test/yzq"

pseudo_df, image_label_dict = generate_pseudo_labels(
    folder_path=folder_path,
    output_csv="yzq_pseudo_labels.csv",
    output_json="yzq_pseudo_labels.json",
    min_confidence=0.15
)

pseudo_df.head()

C:\Users\Luyang Luo\AppData\Roaming\Python\Python38\site-packages\paddle\utils\cpp_extension\extension_utils.py:711: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Creating model: ('PP-LCNet_x1_0_doc_ori', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\Luyang Luo\.paddlex\official_models\PP-LCNet_x1_0_doc_ori`.
Creating model: ('UVDoc', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\Luyang Luo\.paddlex\official_models\UVDoc`.
Creating model: ('PP-LCNet_x1_0_textline_ori', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\Luyang Luo\.paddlex\official_models\PP-LCNet_x1_0_textline_ori`.
C